In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 6 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 calibration.
# - Centre search on the best ACTUALLY observed point.
# - Use a moderate ARD trust region.
# - Compare EI, posterior mean and UCB.
# - Do not automatically expand if acquisition hits boundary.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.44026355, 0.37192991, 0.59085757,
#  0.69024120, 0.13151323]
#
# Predicted:
# mean ≈ -0.209874
# std  ≈ 0.046203
#
# Actual:
# -0.24456987920960416
# ------------------------------------------------------------

week9_pred_mean = -0.209874
week9_pred_std = 0.046203
week9_actual = -0.24456987920960416

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = np.arange(len(X)) != best_idx

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Week 9 calibration improved substantially, so allow
# a moderate region rather than last week's ±0.05 cap.

empirical_cap = min(
    1.5 * nearest_distance,
    0.08
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest point to current best:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.20 * lengthscales,
    0.025,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 10 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Trust-region candidates
# ------------------------------------------------------------

rng = np.random.default_rng(42)

tr_candidates = rng.uniform(
    lower,
    upper,
    size=(350000, 5)
)

tree = cKDTree(X)

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(tr_candidates))


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    tr_candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", tr_candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 11. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", tr_candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from incumbent
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        tr_candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        tr_candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            tr_candidates[idx]
        )
    )


# ------------------------------------------------------------
# 14. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(x[j] - lower[j]) <= tol:
            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(x[j] - upper[j]) <= tol:
            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        tr_candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        tr_candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            tr_candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (29, 5)
Y shape: (29,)

Current best:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.65162863359439

WEEK 9 CALIBRATION CHECK
Predicted mean: -0.209874
Predicted std : 0.046203
Actual        : -0.24456987920960416

Prediction error:
-0.03469587920960415

Error / predicted std:
-0.7509442938684534

GP FIT

Fitted kernel:
1.34**2 * Matern(length_scale=[0.855, 1, 1.38, 0.973, 1.09], nu=2.5) + WhiteKernel(noise_level=0.00282)

ARD lengthscales:
[0.85473205 1.0007297  1.38471421 0.97288137 1.0936664 ]

Normalised inverse-lengthscale sensitivity:
[0.24204533 0.20673305 0.14940548 0.21265069 0.18916545]

EMPIRICAL LOCAL SCALE
Nearest point to current best:
0.0453996159344988

Empirical cap:
0.0680994239017482

WEEK 10 TRUST REGION
Centre:
[0.442929 0.409333 0.63183  0.7398   0.129381]

Half-widths:
[0.06809942 0.06809942 0.06809942 0.06809942 0.06809942]

Lower:
[0.37482958 0.34123358 0.5637

In [2]:
# ============================================================
# FINAL FUNCTION 6 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 calibration improved substantially to about -0.75 sigma.
#
# Highest posterior mean and UCB beta = 0.05, 0.1, 0.25 and
# 0.5 all identify exactly the same candidate.
#
# EI and beta = 1.0 move farther from the incumbent and reward
# additional uncertainty without enough predicted-mean benefit.
#
# Therefore select the stable consensus candidate.

beta = 0.25

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week10_candidate = tr_candidates[final_idx]

print("Week 10 Function 6 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 6 candidate:
[0.44566108 0.40694577 0.5886739  0.70237278 0.12162548]

Predicted mean:
-0.2231541596877067

Predicted std:
0.038525287827434716

UCB:
-0.21352283773084801

Distance from current best:
0.05776293544775942

Portal format:
0.445661-0.406946-0.588674-0.702373-0.121625
